In [ ]:
"""
SafePath — Complete Application (Single-Window Navigation)
"""

import os
import csv
import math
import customtkinter as ctk
import psycopg2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from tkinter import Canvas, messagebox, filedialog, ttk
from PIL import Image, ImageDraw

ctk.set_appearance_mode("light")
ctk.set_default_color_theme("blue")

# ── Colour System
COLOR_SIDEBAR        = "#4B1E0F"
COLOR_SIDEBAR_ACTIVE = "#C47A3A"
COLOR_WORKSPACE_BG   = "#F3EBDD"
COLOR_CARD_BG        = "#EADFC9"
COLOR_CARD_DARK      = "#5A2D0C"
COLOR_CARD_MID       = "#6B3F1D"
COLOR_CARD_LIGHT     = "#D9C2A3"
COLOR_ACCENT_ORANGE  = "#B85C2E"
COLOR_TEXT_DARK      = "#3E2A1F"
COLOR_TEXT_LIGHT     = "#C8A87A"
COLOR_PANEL_BG       = "#EADFC9"
COLOR_GRADIENT_TOP   = "#D4701A"
COLOR_GRADIENT_BOT   = "#2B1005"
COLOR_PROGRESS_TRACK = "#5A2808"
COLOR_PROGRESS_FILL  = "#F5E6D0"
COLOR_INPUT_BG       = "#F5EAD8"

# ── Database
def db_connect():
    return psycopg2.connect(
        dbname="safepath", user="postgres",
        password="cos101", host="localhost", port="5432"
    )

def create_tables():
    conn = db_connect(); cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id       SERIAL PRIMARY KEY,
            username VARCHAR(100) UNIQUE NOT NULL,
            password VARCHAR(100) NOT NULL
        )""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS reports (
            id            SERIAL PRIMARY KEY,
            username      VARCHAR(100) NOT NULL,
            location      VARCHAR(200) NOT NULL,
            incident_type VARCHAR(100) NOT NULL,
            urgency       VARCHAR(50)  NOT NULL,
            description   TEXT,
            anonymous     BOOLEAN      DEFAULT FALSE,
            created_at    TIMESTAMP    DEFAULT NOW()
        )""")
    conn.commit(); cur.close(); conn.close()

def verify_or_register_user(username, password, mode="signin"):
    if not username or not password:
        return False, "Please fill in all fields."
    conn = db_connect(); cur = conn.cursor()
    try:
        cur.execute("SELECT password FROM users WHERE username = %s", (username,))
        row = cur.fetchone()
        if mode == "signin":
            if row is None:          return False, "UserNotFound"
            elif row[0] == password: return True,  "Login successful!"
            else:                    return False, "Incorrect password."
        elif mode == "signup":
            if row is not None:
                return False, "Username already exists."
            cur.execute("INSERT INTO users (username, password) VALUES (%s, %s)",
                        (username, password))
            conn.commit()
            return True, "Account created successfully!"
    except Exception as e:
        return False, str(e)
    finally:
        cur.close(); conn.close()

def add_report(username, location, incident_type, urgency, description, anonymous):
    if not location or not incident_type or not urgency:
        return False, "Please fill in all required fields."
    conn = db_connect(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO reports
                (username, location, incident_type, urgency, description, anonymous)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (username, location, incident_type, urgency, description, anonymous))
        conn.commit()
        return True, "Report submitted successfully."
    except Exception as e:
        return False, str(e)
    finally:
        cur.close(); conn.close()

def fetch_dashboard_metrics():
    conn = db_connect(); cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM reports")
    total = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM reports WHERE urgency ILIKE 'High'")
    high_risk = cur.fetchone()[0]
    cur.execute("""
        SELECT incident_type, location FROM reports
        ORDER BY id DESC LIMIT 5
    """)
    recent = [f"{i} reported in {l}" for i, l in cur.fetchall()]
    cur.close(); conn.close()
    return {"total": total, "high_risk": high_risk,
            "awareness": total + 5, "helped": total, "recent": recent}

def fetch_records(search_text=""):
    conn = db_connect(); cur = conn.cursor()
    val = f"%{search_text}%"
    cur.execute("""
        SELECT id, username, location, incident_type, urgency
        FROM reports
        WHERE username ILIKE %s OR location ILIKE %s OR incident_type ILIKE %s
        ORDER BY id ASC
    """, (val, val, val))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return rows

def export_records_csv(parent_window):
    conn = db_connect(); cur = conn.cursor()
    cur.execute("""
        SELECT id, username, location, incident_type, urgency
        FROM reports ORDER BY id ASC
    """)
    rows = cur.fetchall()
    cur.close(); conn.close()
    path = filedialog.asksaveasfilename(
        parent=parent_window,
        defaultextension=".csv",
        filetypes=[("CSV Files", "*.csv")])
    if path:
        with open(path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["ID", "Username", "Location", "Incident Type", "Urgency"])
            w.writerows(rows)

def fetch_insights_data():
    conn = db_connect(); cur = conn.cursor()
    cur.execute("""
        SELECT incident_type, COUNT(*) FROM reports
        GROUP BY incident_type ORDER BY COUNT(*) DESC
    """)
    rows = cur.fetchall()
    cur.close(); conn.close()
    if rows:
        cats   = [r[0] or "Uncategorised" for r in rows]
        counts = [r[1] for r in rows]
    else:
        cats   = ["No data"]
        counts = [1]
    return {
        "categories": cats, "counts": counts,
        "text_insights": [
            f"📊 Tracking {sum(counts)} profiles across all sectors.",
            "📍 Check Records page for full data splits.",
        ],
    }

# ── Visual helpers (unchanged) ────────────────────────────────────────────────
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def lerp_color(c1, c2, t):
    return tuple(int(c1[i] + (c2[i]-c1[i])*t) for i in range(3))

def rgb_to_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*rgb)

def make_linear_gradient(width, height, top_hex, bot_hex):
    img  = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(img)
    r1,g1,b1 = hex_to_rgb(top_hex)
    r2,g2,b2 = hex_to_rgb(bot_hex)
    for y in range(height):
        t = y / height
        draw.line([(0,y),(width,y)], fill=(
            int(r1+(r2-r1)*t), int(g1+(g2-g1)*t), int(b1+(b2-b1)*t)))
    return ctk.CTkImage(light_image=img, size=(width, height))

# ── Nav items ─────────────────────────────────────────────────────────────────
NAV_ITEMS = [
    ("🏠",  "Dashboard"),
    ("⚠️",  "Situation Check"),
    ("📋",  "Report Case"),
    ("📁",  "Records"),
    ("📊",  "Insights"),
    ("🔗",  "Resources"),
    ("🌐",  "Echoes of Home"),
    ("⚙️",  "Settings"),
]

# ── Risk keywords (unchanged) ─────────────────────────────────────────────────
RISK_KEYWORDS = {
    "cannot leave": 45, "not allowed to leave": 45, "leave": 10,
    "locked": 40, "restricted": 30,
    "passport": 45, "documents": 35, "id withheld": 35,
    "unpaid": 35, "without being paid": 40, "not paid": 35,
    "salary withheld": 35, "working for free": 45,
    "forced to work": 45, "worked for years": 25, "long hours": 20,
    "threat": 30, "threatening": 30, "threatened": 30,
    "fear": 15, "unsafe": 20, "afraid": 20,
    "forced": 35, "coercion": 40,
    "trafficking": 50, "false promise": 40,
    "recruited": 20, "transported": 25,
    "underage": 45, "child labour": 50,
    "isolated": 25, "monitored": 20, "not allowed contact": 35,
}

STORIES_CSV = "peoples_story.csv"
#  ROOT APPLICATION  — single Tk window that hosts every screen

class SafePathApp(ctk.CTk):
    """
    One window. Navigation works by destroying the current screen frame
    and packing a fresh one.  The sidebar (when visible) is rebuilt with
    the correct active-button each time.
    """

    def __init__(self):
        super().__init__()
        self.title("SafePath")
        self.geometry("1440x900")
        self.resizable(True, True)
        self._username = ""
        self._current_frame = None
        # Start with the splash
        self.show_screen("onboarding")

    # ── Core navigator ────────────────────────────────────────────────────────
    def show_screen(self, name, **kwargs):
        """Destroy whatever is on screen and show the requested screen."""
        if self._current_frame is not None:
            self._current_frame.destroy()
            self._current_frame = None

        builders = {
            "onboarding":     self._build_onboarding,
            "login":          self._build_login,
            "Dashboard":      self._build_dashboard,
            "Situation Check":self._build_situation_check,
            "Report Case":    self._build_report_case,
            "Records":        self._build_records,
            "Insights":       self._build_insights,
            "Resources":      self._build_resources,
            "Echoes of Home": self._build_echoes,
            "Settings":       self._build_settings,
        }
        frame = builders[name](**kwargs)
        frame.pack(fill="both", expand=True)
        self._current_frame = frame

    # ── Sidebar builder 
    def _make_sidebar(self, parent, active_item):
        sidebar = ctk.CTkFrame(parent, width=260,
                               fg_color=COLOR_SIDEBAR, corner_radius=0)
        sidebar.pack(side="left", fill="y")
        sidebar.pack_propagate(False)

        ctk.CTkLabel(sidebar, text="SafePath",
                     font=("Georgia", 32, "bold"),
                     text_color="white").pack(pady=(40, 30), anchor="w", padx=30)

        for icon, label in NAV_ITEMS:
            active = label == active_item
            ctk.CTkButton(
                sidebar,
                text=f"{icon}  {label}",
                fg_color=COLOR_SIDEBAR_ACTIVE if active else "transparent",
                hover_color="#8A4F27",
                text_color="white",
                font=("Arial", 14, "bold" if active else "normal"),
                anchor="w", height=40,
                command=lambda l=label: self.show_screen(l),
            ).pack(fill="x", padx=20, pady=5)

        ctk.CTkLabel(sidebar, text="Awareness. Protection. Freedom.",
                     font=("Arial", 11, "italic"),
                     text_color=COLOR_TEXT_LIGHT
                     ).pack(side="bottom", pady=25)
        return sidebar
    #  SCREEN BUILDERS
    # ── Onboarding ────────────────────────────────────────────────────────────
    def _build_onboarding(self, **kw):
        W, H = 1440, 900
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_GRADIENT_BOT,
                                  corner_radius=0)

        c = Canvas(root_frame, width=W, height=H, bd=0,
                   highlightthickness=0, bg=COLOR_GRADIENT_BOT)
        c.place(x=0, y=0, relwidth=1, relheight=1)
        cx, cy = W // 2, H // 2
        max_r  = math.hypot(cx, cy)
        inner  = hex_to_rgb(COLOR_GRADIENT_TOP)
        mid    = hex_to_rgb(COLOR_ACCENT_ORANGE)
        outer  = hex_to_rgb(COLOR_GRADIENT_BOT)
        for i in range(60, -1, -1):
            t  = i / 60
            rx = max_r * t * (W / max_r) * 0.72
            ry = max_r * t * (H / max_r) * 0.72
            col = lerp_color(inner, mid, min(t*2, 1)) if t < 0.5 \
                  else lerp_color(mid, outer, (t-0.5)*2)
            c.create_oval(cx-rx, cy-ry, cx+rx, cy+ry,
                          fill=rgb_to_hex(col), outline="")

        ov = ctk.CTkFrame(root_frame, fg_color="transparent")
        ov.place(relx=0.5, rely=0.5, anchor="center")

        ring = ctk.CTkFrame(ov, width=90, height=90,
                            fg_color=COLOR_SIDEBAR, corner_radius=45,
                            border_color=COLOR_ACCENT_ORANGE, border_width=2)
        ring.pack(pady=(0, 12)); ring.pack_propagate(False)
        ctk.CTkLabel(ring, text="🌿", font=("Segoe UI Emoji", 36),
                     text_color=COLOR_WORKSPACE_BG
                     ).place(relx=0.5, rely=0.5, anchor="center")

        ctk.CTkLabel(ov, text="SafePath",
                     font=("Georgia", 32, "bold"),
                     text_color=COLOR_WORKSPACE_BG).pack(pady=(0, 22))

        BAR_W = 220
        track = ctk.CTkFrame(ov, width=BAR_W, height=6,
                             fg_color=COLOR_PROGRESS_TRACK, corner_radius=3)
        track.pack(pady=(0, 10)); track.pack_propagate(False)
        fill = ctk.CTkFrame(track, width=0, height=6,
                            fg_color=COLOR_PROGRESS_FILL, corner_radius=3)
        fill.place(x=0, y=0)

        MESSAGES = ["Loading…", "Preparing your safe space…",
                    "Almost ready…", "Welcome."]
        lbl = ctk.CTkLabel(ov, text=MESSAGES[0],
                           font=("Segoe UI", 12, "italic"),
                           text_color=COLOR_TEXT_LIGHT)
        lbl.pack()

        def animate(step=0):
            if step > 80:
                lbl.configure(text=MESSAGES[3])
                root_frame.after(700, lambda: self.show_screen("login"))
                return
            t = step / 80
            fill.configure(width=max(int(BAR_W * (1-(1-t)**2)), 1))
            if   step == 0:  lbl.configure(text=MESSAGES[0])
            elif step == 25: lbl.configure(text=MESSAGES[1])
            elif step == 55: lbl.configure(text=MESSAGES[2])
            root_frame.after(32, lambda: animate(step + 1))

        root_frame.after(300, animate)
        return root_frame

    # ── Login ─────────────────────────────────────────────────────────────────
    def _build_login(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color="transparent", corner_radius=0)

        bg = ctk.CTkLabel(root_frame, text="")
        bg.place(x=0, y=0, relwidth=1, relheight=1)
        img = make_linear_gradient(1440, 900, COLOR_ACCENT_ORANGE, COLOR_SIDEBAR)
        bg.configure(image=img); bg.image = img

        current_mode = {"v": "signin"}

        card = ctk.CTkFrame(root_frame, width=420, height=480,
                            fg_color=COLOR_CARD_BG, corner_radius=24,
                            border_width=1, border_color="#D9C2A3")
        card.place(relx=0.5, rely=0.5, anchor="center")
        card.pack_propagate(False)

        title = ctk.CTkLabel(card, text="Login",
                             font=("Georgia", 32, "bold"),
                             text_color=COLOR_SIDEBAR)
        title.pack(pady=(45, 30))

        user_entry = ctk.CTkEntry(card, placeholder_text="Username",
                                  width=320, height=45, corner_radius=10,
                                  fg_color=COLOR_WORKSPACE_BG,
                                  text_color=COLOR_TEXT_DARK,
                                  border_color="#D9C2A3")
        user_entry.pack(pady=12)

        pass_entry = ctk.CTkEntry(card, placeholder_text="Password",
                                  show="•", width=320, height=45,
                                  corner_radius=10,
                                  fg_color=COLOR_WORKSPACE_BG,
                                  text_color=COLOR_TEXT_DARK,
                                  border_color="#D9C2A3")
        pass_entry.pack(pady=12)

        def do_auth():
            u = user_entry.get().strip()
            p = pass_entry.get().strip()
            ok, msg = verify_or_register_user(u, p, mode=current_mode["v"])
            if ok:
                messagebox.showinfo("SafePath", msg)
                self._username = u
                self._password = p  # <-- Add this line to remember the password
                self.show_screen("Dashboard")
            else:
                if msg == "UserNotFound":
                    if messagebox.askyesno("User Not Found",
                                           f"No account for '{u}'.\nCreate one?"):
                        switch()
                else:
                    messagebox.showerror("Auth Error", msg)
        def switch():
            current_mode["v"] = "signup" if current_mode["v"] == "signin" else "signin"
            if current_mode["v"] == "signin":
                title.configure(text="Login")
                btn.configure(text="Login", fg_color=COLOR_SIDEBAR)
                toggle.configure(text="Don't have an account? Sign up here")
            else:
                title.configure(text="Create Account")
                btn.configure(text="Register Account",
                              fg_color=COLOR_ACCENT_ORANGE)
                toggle.configure(text="Already have an account? Sign in here")

        btn = ctk.CTkButton(card, text="Login", command=do_auth,
                            fg_color=COLOR_SIDEBAR, hover_color="#33140A",
                            text_color="white", width=320, height=48,
                            corner_radius=12,
                            font=("Georgia", 16, "bold"))
        btn.pack(pady=(25, 15))

        toggle = ctk.CTkLabel(card,
                              text="Don't have an account? Sign up here",
                              font=("Segoe UI", 12, "underline"),
                              text_color=COLOR_TEXT_DARK, cursor="hand2")
        toggle.pack(pady=5)
        toggle.bind("<Button-1>", lambda e: switch())

        return root_frame

    # ── Dashboard ─────────────────────────────────────────────────────────────
    def _build_dashboard(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Dashboard")

        main = ctk.CTkFrame(root_frame, fg_color=COLOR_WORKSPACE_BG,
                            corner_radius=0)
        main.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(main, text="Dashboard",
                     font=("Georgia", 38, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=40, pady=(40, 5))
        ctk.CTkLabel(main,
                     text=f"Welcome back, {self._username}.",
                     font=("Arial", 16),
                     text_color="#5E4B3C"
                     ).pack(anchor="nw", padx=40, pady=(0, 25))

        grid = ctk.CTkFrame(main, fg_color="transparent")
        grid.pack(fill="both", expand=True, padx=40, pady=(0, 40))
        grid.columnconfigure(0, weight=1); grid.columnconfigure(1, weight=1)
        grid.rowconfigure(0, weight=1);   grid.rowconfigure(1, weight=1)
        grid.rowconfigure(2, weight=2)

        def scard(row, col, bg, label, lc, vc, pl=0, pr=0):
            f = ctk.CTkFrame(grid, fg_color=bg, corner_radius=18)
            f.grid(row=row, column=col, sticky="nsew",
                   padx=(pl, pr), pady=10)
            f.pack_propagate(False)
            ctk.CTkLabel(f, text=label, font=("Arial", 16),
                         text_color=lc).pack(anchor="nw", padx=25, pady=(25, 5))
            v = ctk.CTkLabel(f, text="--",
                             font=("Georgia", 42, "bold"), text_color=vc)
            v.pack(anchor="nw", padx=25)
            return v

        v_total    = scard(0, 0, COLOR_CARD_DARK,     "Total Reports",
                           "#F3EBDD", "white",          pr=15)
        v_highrisk = scard(0, 1, COLOR_ACCENT_ORANGE, "High Risk Flagged",
                           COLOR_SIDEBAR, COLOR_SIDEBAR, pl=15)
        v_aware    = scard(1, 0, COLOR_CARD_LIGHT,    "Awareness Check",
                           COLOR_SIDEBAR, COLOR_SIDEBAR, pr=15)
        v_helped   = scard(1, 1, COLOR_CARD_MID,      "People Helped",
                           "#F3EBDD", "white",          pl=15)

        # CTA panel
        cta = ctk.CTkFrame(grid, fg_color=COLOR_CARD_DARK, corner_radius=18)
        cta.grid(row=2, column=0, sticky="nsew", padx=(0, 15), pady=(15, 0))
        cta.pack_propagate(False)
        ctk.CTkLabel(cta, text="Unsure about a situation?",
                     font=("Georgia", 26, "bold"),
                     text_color="#F3EBDD").pack(pady=(35, 10))
        ctk.CTkLabel(cta,
                     text="Use our AI-driven analysis tool to identify\n"
                          "signs of exploitation or forced labour.",
                     font=("Arial", 15), text_color="#D9C2A3",
                     justify="center").pack(pady=(0, 25))
        ctk.CTkButton(cta, text="Start Risk Assessment",
                      fg_color=COLOR_ACCENT_ORANGE, hover_color="#A44E25",
                      text_color="white", font=("Arial", 15, "bold"),
                      height=45, corner_radius=10,
                      command=lambda: self.show_screen("Situation Check")
                      ).pack(ipadx=20)

        # Recent reports panel
        rec = ctk.CTkFrame(grid, fg_color=COLOR_CARD_BG, corner_radius=18,
                           border_width=1, border_color="#D9C2A3")
        rec.grid(row=2, column=1, sticky="nsew", padx=(15, 0), pady=(15, 0))
        rec.pack_propagate(False)
        ctk.CTkLabel(rec, text="Recent Reports",
                     font=("Georgia", 22, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=25, pady=(25, 10))
        feed = ctk.CTkFrame(rec, fg_color="transparent")
        feed.pack(fill="both", expand=True, padx=25)

        def refresh():
            # guard: if the frame was destroyed, stop refreshing
            if not root_frame.winfo_exists():
                return
            data = fetch_dashboard_metrics()
            v_total.configure(text=str(data["total"]))
            v_highrisk.configure(text=str(data["high_risk"]))
            v_aware.configure(text=str(data["awareness"]))
            v_helped.configure(text=str(data["helped"]))
            for w in feed.winfo_children():
                w.destroy()
            if data["recent"]:
                for t in data["recent"]:
                    ctk.CTkLabel(feed, text=f"• {t}",
                                 font=("Arial", 14),
                                 text_color=COLOR_TEXT_DARK,
                                 anchor="w").pack(fill="x", pady=6)
                    ctk.CTkFrame(feed, height=1,
                                 fg_color="#D9C2A3").pack(fill="x", pady=(2, 6))
            else:
                ctk.CTkLabel(feed, text="No notifications yet.",
                             font=("Arial", 14, "italic"),
                             text_color="#766555").pack(pady=20)
            root_frame.after(5000, refresh)

        refresh()
        return root_frame

    # ── Situation Check ───────────────────────────────────────────────────────
    def _build_situation_check(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Situation Check")

        main = ctk.CTkFrame(root_frame, fg_color=COLOR_WORKSPACE_BG,
                            corner_radius=0)
        main.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(main, text="Situation Checker",
                     font=("Georgia", 36, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=40, pady=(40, 5))
        ctk.CTkLabel(main,
                     text="Analyse suspicious activity for indicators of modern slavery.",
                     font=("Arial", 15), text_color="#5E4B3C"
                     ).pack(anchor="nw", padx=40, pady=(0, 30))

        inp = ctk.CTkFrame(main, fg_color=COLOR_CARD_BG, border_width=2,
                           border_color="#8A4F27", corner_radius=15, height=180)
        inp.pack(fill="x", padx=40)
        inp.pack_propagate(False)

        textbox = ctk.CTkTextbox(inp, height=100, fg_color="#E9D8BE",
                                 border_color="#8A4F27", border_width=1,
                                 text_color=COLOR_TEXT_DARK,
                                 font=("Arial", 14))
        textbox.pack(fill="x", padx=30, pady=(25, 15))
        textbox.insert("1.0", "Describe the situation in detail…")

        bot = ctk.CTkFrame(main, fg_color="transparent")
        bot.pack(fill="both", expand=True, padx=40, pady=30)

        meter = ctk.CTkFrame(bot, fg_color=COLOR_CARD_BG, border_width=2,
                             border_color="#8A4F27", corner_radius=15)
        meter.pack(side="left", fill="both", expand=True)
        meter.pack_propagate(False)

        gauge = ctk.CTkCanvas(meter, width=220, height=220,
                              bg=COLOR_CARD_BG, highlightthickness=0)
        gauge.pack(pady=(30, 10))
        gauge.create_arc(20, 20, 200, 200, start=30, extent=300,
                         style="arc", width=2)
        gauge.create_line(110, 110, 160, 70, width=3)
        gauge.create_oval(95, 95, 125, 125, fill=COLOR_WORKSPACE_BG)

        result_lbl = ctk.CTkLabel(meter, text="Awaiting Analysis",
                                  font=("Georgia", 28, "bold"),
                                  text_color="#5E4B3C")
        result_lbl.pack(pady=20)

        right = ctk.CTkFrame(bot, width=280, fg_color=COLOR_CARD_MID,
                             corner_radius=15)
        right.pack(side="right", fill="y", padx=(20, 0))
        right.pack_propagate(False)

        ctk.CTkLabel(right, text="Common Indicators",
                     font=("Georgia", 20, "bold"),
                     text_color="white").pack(anchor="w", padx=20, pady=(25, 20))
        for emoji, label in [("🔴","High Risk"), ("🟠","Caution"), ("🟢","Low Risk")]:
            ctk.CTkLabel(right, text=f"{emoji}  {label}",
                         font=("Arial", 14),
                         text_color="white").pack(anchor="w", padx=20, pady=5)

        indicators_lbl = ctk.CTkLabel(right, text="No indicators yet.",
                                      justify="left", font=("Arial", 13),
                                      text_color="#F3EBDD", wraplength=240)
        indicators_lbl.pack(anchor="w", padx=20, pady=(30, 10))

        def analyse():
            text  = textbox.get("1.0", "end").lower()
            score = 0; found = []
            for kw, pts in RISK_KEYWORDS.items():
                if kw in text:
                    score += pts; found.append(kw)
            if score >= 70:   level, color = "HIGH RISK", "red"
            elif score >= 35: level, color = "CAUTION",   "orange"
            else:             level, color = "LOW RISK",  "green"
            result_lbl.configure(text=level, text_color=color)
            indicators_lbl.configure(
                text=("Indicators Found:\n• " + "\n• ".join(found)) if found
                     else "No major indicators detected.")

        ctk.CTkButton(inp, text="Run Risk Analysis", command=analyse,
                      fg_color="#8A4F27", hover_color="#A44E25",
                      text_color="white", width=180, height=40,
                      corner_radius=10).pack(anchor="e", padx=30)

        return root_frame

    # ── Report Case ───────────────────────────────────────────────────────────
    def _build_report_case(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Report Case")

        scroll = ctk.CTkScrollableFrame(root_frame,
                                        fg_color=COLOR_WORKSPACE_BG,
                                        scrollbar_button_color=COLOR_CARD_LIGHT,
                                        scrollbar_button_hover_color=COLOR_ACCENT_ORANGE)
        scroll.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(scroll, text="Report Case",
                     font=("Georgia", 34, "bold"),
                     text_color=COLOR_TEXT_DARK,
                     anchor="w").pack(anchor="w", padx=40, pady=(36, 4))
        ctk.CTkLabel(scroll, text="All fields marked * are required.",
                     font=("Arial", 13),
                     text_color=COLOR_TEXT_LIGHT,
                     anchor="w").pack(anchor="w", padx=40, pady=(0, 28))

        card = ctk.CTkFrame(scroll, fg_color=COLOR_CARD_BG, corner_radius=18,
                            border_width=1, border_color="#D9C2A3")
        card.pack(fill="x", padx=40, pady=(0, 32))

        def sec(txt):
            ctk.CTkLabel(card, text=txt, font=("Georgia", 15, "bold"),
                         text_color=COLOR_TEXT_DARK,
                         anchor="w").pack(anchor="w", padx=28, pady=(22, 6))

        def flbl(parent, txt):
            ctk.CTkLabel(parent, text=txt, font=("Arial", 12),
                         text_color=COLOR_TEXT_DARK,
                         anchor="w").pack(anchor="w", pady=(0, 4))

        INCIDENT_OPTIONS = ["Theft","Assault","Harassment","Forced Labour",
                            "Human Trafficking","Domestic Abuse","Other"]
        URGENCY_OPTIONS  = ["Low","Medium","High","Critical"]

        sec("Incident Details")
        r1 = ctk.CTkFrame(card, fg_color="transparent")
        r1.pack(fill="x", padx=28, pady=(0, 12))
        r1.columnconfigure(0, weight=1); r1.columnconfigure(1, weight=1)

        l1 = ctk.CTkFrame(r1, fg_color="transparent")
        l1.grid(row=0, column=0, sticky="ew", padx=(0, 12))
        flbl(l1, "Location *")
        loc = ctk.CTkEntry(l1, placeholder_text="e.g. Lagos, Nigeria",
                           height=42, fg_color=COLOR_WORKSPACE_BG,
                           border_color="#D9C2A3", text_color=COLOR_TEXT_DARK)
        loc.pack(fill="x")

        r1r = ctk.CTkFrame(r1, fg_color="transparent")
        r1r.grid(row=0, column=1, sticky="ew", padx=(12, 0))
        flbl(r1r, "Incident Type *")
        inc = ctk.CTkOptionMenu(r1r, values=INCIDENT_OPTIONS,
                                fg_color=COLOR_WORKSPACE_BG,
                                button_color="#D9C2A3",
                                button_hover_color=COLOR_ACCENT_ORANGE,
                                text_color=COLOR_TEXT_DARK,
                                dropdown_fg_color=COLOR_WORKSPACE_BG,
                                dropdown_text_color=COLOR_TEXT_DARK,
                                dropdown_hover_color=COLOR_CARD_LIGHT,
                                font=("Arial", 13), height=42)
        inc.set(INCIDENT_OPTIONS[0]); inc.pack(fill="x")

        r2 = ctk.CTkFrame(card, fg_color="transparent")
        r2.pack(fill="x", padx=28, pady=(0, 12))
        r2.columnconfigure(0, weight=1); r2.columnconfigure(1, weight=1)

        l2 = ctk.CTkFrame(r2, fg_color="transparent")
        l2.grid(row=0, column=0, sticky="ew", padx=(0, 12))
        flbl(l2, "Urgency Level *")
        urg = ctk.CTkOptionMenu(l2, values=URGENCY_OPTIONS,
                                fg_color=COLOR_WORKSPACE_BG,
                                button_color="#D9C2A3",
                                button_hover_color=COLOR_ACCENT_ORANGE,
                                text_color=COLOR_TEXT_DARK,
                                dropdown_fg_color=COLOR_WORKSPACE_BG,
                                dropdown_text_color=COLOR_TEXT_DARK,
                                dropdown_hover_color=COLOR_CARD_LIGHT,
                                font=("Arial", 13), height=42)
        urg.set("Medium"); urg.pack(fill="x")

        r2r = ctk.CTkFrame(r2, fg_color="transparent")
        r2r.grid(row=0, column=1, sticky="ew", padx=(12, 0))
        flbl(r2r, "Submit Anonymously")
        import tkinter as _tk
        anon_var = _tk.BooleanVar(value=False)
        ctk.CTkSwitch(r2r, text="Hide my identity",
                      variable=anon_var, font=("Arial", 12),
                      text_color=COLOR_TEXT_DARK,
                      fg_color=COLOR_CARD_LIGHT,
                      progress_color=COLOR_ACCENT_ORANGE,
                      button_color=COLOR_SIDEBAR,
                      button_hover_color=COLOR_CARD_DARK
                      ).pack(anchor="w", pady=(8, 0))

        sec("Description")
        desc = ctk.CTkTextbox(card, height=160,
                              fg_color=COLOR_WORKSPACE_BG,
                              border_color="#D9C2A3", border_width=1,
                              text_color=COLOR_TEXT_DARK, corner_radius=10)
        desc.pack(fill="x", padx=28, pady=(0, 28))

        status_lbl = ctk.CTkLabel(scroll, text="",
                                  font=("Arial", 13), anchor="center")
        status_lbl.pack(pady=(0, 8))

        def submit():
            anon  = anon_var.get()
            uname = "Anonymous" if anon else self._username
            ok, msg = add_report(uname, loc.get().strip(), inc.get(),
                                 urg.get(), desc.get("1.0","end").strip(), anon)
            if ok:
                loc.delete(0, "end")
                inc.set(INCIDENT_OPTIONS[0]); urg.set("Medium")
                desc.delete("1.0", "end")
                status_lbl.configure(text="✅  Report submitted!",
                                     text_color="#2E7D32")
                scroll.after(2000, lambda: self.show_screen("Dashboard"))
            else:
                status_lbl.configure(text=f"❌  {msg}", text_color="#B71C1C")

        ctk.CTkButton(scroll, text="📋  Submit Report", command=submit,
                      fg_color=COLOR_ACCENT_ORANGE, hover_color="#A44E25",
                      text_color="white", font=("Georgia", 15, "bold"),
                      height=50, corner_radius=12
                      ).pack(fill="x", padx=40, pady=(0, 40))

        return root_frame

    # ── Records ───────────────────────────────────────────────────────────────
    def _build_records(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Records")

        main = ctk.CTkFrame(root_frame, fg_color=COLOR_WORKSPACE_BG,
                            corner_radius=0)
        main.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(main, text="Records",
                     font=("Georgia", 32, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=40, pady=(30, 10))

        bar = ctk.CTkFrame(main, fg_color="transparent")
        bar.pack(fill="x", padx=40, pady=(10, 0))

        search_entry = ctk.CTkEntry(bar, width=300,
                                    placeholder_text="Search records…",
                                    fg_color=COLOR_INPUT_BG,
                                    border_color=COLOR_CARD_LIGHT,
                                    text_color=COLOR_TEXT_DARK)
        search_entry.pack(side="left", padx=(0, 10))

        tbl_frame = ctk.CTkFrame(main, fg_color="#F7F0E4")
        tbl_frame.pack(fill="both", expand=True, padx=40, pady=30)

        style = ttk.Style()
        style.theme_use("default")
        style.configure("SafePath.Treeview",
                        background="#F3E7D7", foreground=COLOR_TEXT_DARK,
                        fieldbackground="#F3E7D7", rowheight=35,
                        font=("Arial", 11))
        style.configure("SafePath.Treeview.Heading",
                        background=COLOR_CARD_DARK, foreground="white",
                        font=("Georgia", 12, "bold"))
        style.map("SafePath.Treeview",
                  background=[("selected", COLOR_SIDEBAR_ACTIVE)])

        cols = ("ID", "Username", "Location", "Incident Type", "Urgency")
        tree = ttk.Treeview(tbl_frame, columns=cols,
                            show="headings", height=20,
                            style="SafePath.Treeview")
        for col in cols:
            tree.heading(col, text=col)
            tree.column(col, width=180)
        tree.pack(fill="both", expand=True, padx=20, pady=20)

        def load():
            rows = fetch_records(search_entry.get())
            for r in tree.get_children(): tree.delete(r)
            for row in rows: tree.insert("", "end", values=row)

        ctk.CTkButton(bar, text="Search", command=load,
                      fg_color=COLOR_CARD_DARK, hover_color="#7A431A",
                      text_color="white", width=90
                      ).pack(side="left", padx=(0, 10))
        ctk.CTkButton(bar, text="Export as CSV",
                      command=lambda: export_records_csv(root_frame),
                      fg_color=COLOR_CARD_DARK, hover_color="#7A431A",
                      text_color="white"
                      ).pack(side="right")

        load()
        return root_frame

    # ── Insights ──────────────────────────────────────────────────────────────
    def _build_insights(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Insights")

        main = ctk.CTkFrame(root_frame, fg_color=COLOR_WORKSPACE_BG,
                            corner_radius=0)
        main.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(main, text="Analytics & Insights",
                     font=("Georgia", 32, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=40, pady=(25, 5))
        ctk.CTkLabel(main,
                     text="Review vulnerability trends and systemic data distributions.",
                     font=("Arial", 14), text_color="#5E4B3C"
                     ).pack(anchor="nw", padx=40, pady=(0, 20))

        grid = ctk.CTkFrame(main, fg_color="transparent")
        grid.pack(fill="both", expand=True, padx=40, pady=(0, 30))
        grid.columnconfigure(0, weight=1); grid.columnconfigure(1, weight=1)
        grid.rowconfigure(0, weight=1)

        data   = fetch_insights_data()
        colors = [COLOR_ACCENT_ORANGE, COLOR_CARD_DARK, COLOR_CARD_MID,
                  COLOR_CARD_LIGHT, "#A44E25", "#766555"]

        # Left: takeaways + donut
        left = ctk.CTkFrame(grid, fg_color=COLOR_PANEL_BG, corner_radius=18,
                            border_width=1, border_color="#D9C2A3")
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 15))

        ctk.CTkLabel(left, text="System Key Takeaways",
                     font=("Georgia", 18, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="nw", padx=20, pady=(15, 10))
        for insight in data["text_insights"]:
            box = ctk.CTkFrame(left, fg_color=COLOR_WORKSPACE_BG, corner_radius=8)
            box.pack(fill="x", padx=20, pady=4)
            ctk.CTkLabel(box, text=insight, font=("Arial", 12),
                         text_color=COLOR_TEXT_DARK, justify="left",
                         wraplength=420).pack(anchor="w", padx=12, pady=8)

        donut_c = ctk.CTkFrame(left, fg_color=COLOR_PANEL_BG, corner_radius=14)
        donut_c.pack(fill="both", expand=True, padx=20, pady=(10, 15))
        fig1, ax1 = plt.subplots(figsize=(4.5, 3.5), facecolor=COLOR_PANEL_BG)
        ax1.set_facecolor(COLOR_PANEL_BG)
        wedges, _, autotexts = ax1.pie(
            data["counts"], autopct="%1.0f%%", startangle=90,
            colors=colors[:len(data["counts"])],
            pctdistance=0.72,
            textprops=dict(color="white", fontsize=10, weight="bold"),
            wedgeprops=dict(width=0.38, edgecolor=COLOR_PANEL_BG, linewidth=2.5))
        ax1.set_title("Percentage Breakdown", fontsize=12, fontweight="bold",
                      color=COLOR_TEXT_DARK, pad=5)
        ax1.legend(wedges, data["categories"], loc="center left",
                   bbox_to_anchor=(0.95, 0.5), frameon=False,
                   fontsize=9, labelcolor=COLOR_TEXT_DARK)
        plt.tight_layout()
        FigureCanvasTkAgg(fig1, master=donut_c).get_tk_widget(
        ).pack(fill="both", expand=True)

        # Right: horizontal bar chart
        right = ctk.CTkFrame(grid, fg_color=COLOR_PANEL_BG, corner_radius=18,
                             border_width=1, border_color="#D9C2A3")
        right.grid(row=0, column=1, sticky="nsew", padx=(15, 0))
        fig2, ax2 = plt.subplots(figsize=(6, 6), facecolor=COLOR_PANEL_BG)
        ax2.set_facecolor(COLOR_PANEL_BG)
        bars = ax2.barh(data["categories"], data["counts"],
                        color=COLOR_ACCENT_ORANGE,
                        edgecolor=COLOR_CARD_DARK, height=0.55)
        ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
        ax2.spines["left"].set_color(COLOR_TEXT_DARK)
        ax2.spines["bottom"].set_color(COLOR_TEXT_DARK)
        ax2.tick_params(colors=COLOR_TEXT_DARK, labelsize=11)
        ax2.set_title("Incidents by Category", fontsize=14, fontweight="bold",
                      color=COLOR_TEXT_DARK, pad=15)
        for bar in bars:
            w = bar.get_width()
            ax2.annotate(f" {int(w)}",
                         xy=(w, bar.get_y() + bar.get_height() / 2),
                         xytext=(3, 0), textcoords="offset points",
                         ha="left", va="center", fontsize=11,
                         color=COLOR_TEXT_DARK, weight="bold")
        plt.tight_layout()
        FigureCanvasTkAgg(fig2, master=right).get_tk_widget(
        ).pack(fill="both", expand=True, padx=20, pady=20)

        return root_frame

    # ── Resources ─────────────────────────────────────────────────────────────
    def _build_resources(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Resources")

        content = ctk.CTkScrollableFrame(root_frame,
                                         fg_color=COLOR_WORKSPACE_BG)
        content.pack(side="left", fill="both", expand=True)

        ctk.CTkLabel(content, text="Resources & Support",
                     font=("Georgia", 30, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="w", padx=40, pady=(35, 5))
        ctk.CTkLabel(content,
                     text="Emergency contacts, NGOs and support systems.",
                     font=("Arial", 14),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="w", padx=40, pady=(0, 25))

        emergency = ctk.CTkFrame(content, fg_color=COLOR_CARD_BG,
                                 corner_radius=16)
        emergency.pack(fill="x", padx=40, pady=(0, 25))
        ctk.CTkLabel(emergency, text="Emergency Helpline",
                     font=("Arial", 18, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="w", padx=20, pady=(20, 10))
        ctk.CTkLabel(emergency,
                     text="07030000203\n08002255627847",
                     font=("Arial", 20, "bold"),
                     text_color=COLOR_ACCENT_ORANGE,
                     justify="center").pack(pady=10)
        ctk.CTkButton(emergency, text="📞 CALL NOW",
                      fg_color=COLOR_ACCENT_ORANGE,
                      hover_color="#8A4F27", height=45,
                      command=lambda: messagebox.showinfo(
                          "Emergency", "Calling NAPTIP emergency support...")
                      ).pack(pady=(5, 20))

        ngo = ctk.CTkFrame(content, fg_color=COLOR_CARD_BG, corner_radius=16)
        ngo.pack(fill="x", padx=40, pady=(0, 40))
        ctk.CTkLabel(ngo, text="Support Organisations",
                     font=("Arial", 18, "bold"),
                     text_color=COLOR_TEXT_DARK
                     ).pack(anchor="w", padx=20, pady=(20, 20))
        for name, details in [
            ("Devatop Centre for African Development", "Abuja • +234 903 000 2362"),
            ("WOCON",  "Lagos • +234 803 719 0133"),
            ("NAPTIP", "07030000203"),
        ]:
            c = ctk.CTkFrame(ngo, fg_color=COLOR_WORKSPACE_BG, corner_radius=12)
            c.pack(fill="x", padx=20, pady=10)
            ctk.CTkLabel(c, text=name, font=("Arial", 15, "bold"),
                         text_color=COLOR_TEXT_DARK
                         ).pack(anchor="w", padx=15, pady=(12, 4))
            ctk.CTkLabel(c, text=details, font=("Arial", 13),
                         text_color=COLOR_TEXT_DARK
                         ).pack(anchor="w", padx=15, pady=(0, 12))

        return root_frame

    # ── Echoes of Home
    def _build_echoes(self, **kw):
        PLACEHOLDER = "Share your story here…"

        # ensure CSV exists
        if not os.path.exists(STORIES_CSV):
            with open(STORIES_CSV, "w", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow(["Name", "Country", "Story"])

        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG,
                                  corner_radius=0)
        self._make_sidebar(root_frame, "Echoes of Home")

        # ── Main form view 
        main_c = ctk.CTkScrollableFrame(root_frame, fg_color="transparent")

        ctk.CTkLabel(main_c, text="Echoes of Home",
                     font=("Georgia", 28, "bold"),
                     text_color=COLOR_TEXT_DARK).pack(anchor="w", pady=(0, 2))
        ctk.CTkLabel(main_c,
                     text="Exploring stories of displacement, resilience, and cultural reconnection.",
                     font=("Arial", 14),
                     text_color="#6E5D53").pack(anchor="w", pady=(0, 25))

        cards_row = ctk.CTkFrame(main_c, fg_color="transparent")
        cards_row.pack(fill="x", pady=(0, 30))
        for i in range(3):
            cards_row.columnconfigure(i, weight=1, uniform="card")
            card = ctk.CTkFrame(cards_row, height=140,
                                fg_color=COLOR_CARD_BG, corner_radius=15)
            card.grid(row=0, column=i, padx=5, sticky="ew")
            card.grid_propagate(False)
            fname = f"Rectangle 2{5+i}.png"
            try:
                img = ctk.CTkImage(light_image=Image.open(fname), size=(230, 140))
                ctk.CTkLabel(card, text="", image=img).pack(fill="both", expand=True)
            except FileNotFoundError:
                ctk.CTkLabel(card, text=f"Image {i+1}",
                             text_color=COLOR_TEXT_LIGHT).pack(expand=True)

        form = ctk.CTkFrame(main_c, fg_color=COLOR_CARD_BG,
                            border_color="#8B5A2B", border_width=1,
                            corner_radius=15)
        form.pack(fill="both", expand=True, pady=10, ipady=20)

        ctk.CTkLabel(form, text="Share Your Story",
                     font=("Georgia", 20, "bold"),
                     text_color=COLOR_SIDEBAR).pack(pady=(20, 15))

        inp_row = ctk.CTkFrame(form, fg_color="transparent")
        inp_row.pack(fill="x", padx=30, pady=5)
        inp_row.columnconfigure(0, weight=1); inp_row.columnconfigure(1, weight=1)

        name_e = ctk.CTkEntry(inp_row, placeholder_text="Name",
                              fg_color=COLOR_INPUT_BG, border_width=0,
                              height=40, corner_radius=8,
                              text_color=COLOR_TEXT_DARK)
        name_e.grid(row=0, column=0, padx=(0, 10), sticky="ew")

        country_e = ctk.CTkEntry(inp_row, placeholder_text="Country",
                                 fg_color=COLOR_INPUT_BG, border_width=0,
                                 height=40, corner_radius=8,
                                 text_color=COLOR_TEXT_DARK)
        country_e.grid(row=0, column=1, padx=(10, 0), sticky="ew")

        story_box = ctk.CTkTextbox(form, font=("Georgia", 13, "italic"),
                                   fg_color=COLOR_INPUT_BG, border_width=0,
                                   height=150, corner_radius=8,
                                   text_color=COLOR_TEXT_LIGHT)
        story_box.pack(fill="x", padx=30, pady=15)
        story_box.insert("1.0", PLACEHOLDER)

        def clear_ph(_):
            if story_box.get("1.0", "end-1c").strip() == PLACEHOLDER:
                story_box.delete("1.0", "end")
                story_box.configure(text_color=COLOR_TEXT_DARK)

        def restore_ph(_):
            if not story_box.get("1.0", "end-1c").strip():
                story_box.insert("1.0", PLACEHOLDER)
                story_box.configure(text_color=COLOR_TEXT_LIGHT)

        story_box.bind("<FocusIn>",  clear_ph)
        story_box.bind("<FocusOut>", restore_ph)

        err_lbl = ctk.CTkLabel(form, text="",
                               font=("Arial", 12, "bold"),
                               text_color="#A02D2D")
        err_lbl.pack(pady=(0, 5))

        # ── Confirmation view ─────────────────────────────────────────────────
        conf_c = ctk.CTkFrame(root_frame, fg_color="transparent")
        inner  = ctk.CTkFrame(conf_c, fg_color=COLOR_CARD_BG,
                              border_color="#8B5A2B", border_width=1,
                              corner_radius=15)
        inner.pack(expand=True, padx=40, pady=100, ipady=40, ipadx=40)
        ctk.CTkLabel(inner, text="Thank You!",
                     font=("Georgia", 32, "bold"),
                     text_color=COLOR_SIDEBAR).pack(pady=(40, 15))
        conf_msg = ctk.CTkLabel(inner, text="",
                                font=("Arial", 15), text_color="#6E5D53",
                                justify="center")
        conf_msg.pack(pady=(0, 30), padx=20)

        def show_main():
            conf_c.pack_forget()
            main_c.pack(side="left", fill="both", expand=True, padx=40, pady=30)

        ctk.CTkButton(inner, text="Submit Another Story",
                      command=show_main,
                      font=("Arial", 14, "bold"),
                      fg_color=COLOR_SIDEBAR,
                      hover_color="#5D3216",
                      text_color=COLOR_CARD_BG,
                      height=40, width=220, corner_radius=20
                      ).pack(pady=(10, 20))

        def submit_story():
            name    = name_e.get().strip()
            country = country_e.get().strip()
            story   = story_box.get("1.0", "end-1c").strip()
            if not name or not country or not story or story == PLACEHOLDER:
                err_lbl.configure(text="Please fill out all fields.")
                return
            try:
                with open(STORIES_CSV, "a", newline="", encoding="utf-8") as f:
                    csv.writer(f).writerow(
                        [name, country, story.replace("\n", " ")])
            except Exception as e:
                err_lbl.configure(text=f"Save error: {e}")
                return
            conf_msg.configure(
                text=f"Thank you, {name}!\n"
                     f"Your story from {country} has been logged.\n\n"
                     "We appreciate your contribution.")
            name_e.delete(0, "end")
            country_e.delete(0, "end")
            story_box.delete("1.0", "end")
            story_box.insert("1.0", PLACEHOLDER)
            story_box.configure(text_color=COLOR_TEXT_LIGHT)
            err_lbl.configure(text="")
            main_c.pack_forget()
            conf_c.pack(side="left", fill="both", expand=True)

        ctk.CTkButton(form, text="Submit Story", command=submit_story,
                      font=("Arial", 14, "bold"),
                      fg_color=COLOR_SIDEBAR,
                      hover_color="#5D3216",
                      text_color=COLOR_CARD_BG,
                      height=40, width=200, corner_radius=20
                      ).pack(pady=(10, 10))

        show_main()
        return root_frame

    # ── Settings ──────────────────────────────────────────────────────────────
    # ── Settings ──────────────────────────────────────────────────────────────
    def _build_settings(self, **kw):
        root_frame = ctk.CTkFrame(self, fg_color=COLOR_WORKSPACE_BG, corner_radius=0)
        self._make_sidebar(root_frame, "Settings")

        scroll = ctk.CTkScrollableFrame(root_frame,
                                        fg_color=COLOR_WORKSPACE_BG,
                                        scrollbar_button_color=COLOR_CARD_LIGHT,
                                        scrollbar_button_hover_color=COLOR_ACCENT_ORANGE)
        scroll.pack(side="right", fill="both", expand=True)

        ctk.CTkLabel(scroll, text="Settings",
                     font=("Georgia", 26, "bold"),
                     text_color=COLOR_TEXT_DARK,
                     anchor="w").pack(anchor="w", padx=32, pady=(28, 2))
        
        ctk.CTkLabel(scroll,
                     text="Manage your account, privacy, notification and preferences.",
                     font=("Segoe UI", 12),
                     text_color=COLOR_TEXT_DARK,
                     anchor="w").pack(anchor="w", padx=32, pady=(0, 20))

        # ── Account Credentials Card ──────────────────────────────────────────
        acct_card = ctk.CTkFrame(scroll, fg_color=COLOR_CARD_BG, corner_radius=12,
                                 border_width=1, border_color="#D9C2A3")
        acct_card.pack(fill="x", padx=32, pady=(0, 16))
        
        ctk.CTkLabel(acct_card, text="👤  Account Details", 
                     font=("Segoe UI", 14, "bold"),
                     text_color=COLOR_TEXT_DARK, anchor="w").pack(anchor="w", padx=16, pady=(14, 10))
        
        # Username Row
        u_frame = ctk.CTkFrame(acct_card, fg_color="transparent")
        u_frame.pack(fill="x", padx=16, pady=4)
        ctk.CTkLabel(u_frame, text="Username: ", font=("Segoe UI", 12, "bold"), text_color=COLOR_TEXT_DARK).pack(side="left")
        ctk.CTkLabel(u_frame, text=f"{self._username}", font=("Segoe UI", 12), text_color=COLOR_CARD_DARK).pack(side="left")
        
        # Password Row
        p_frame = ctk.CTkFrame(acct_card, fg_color="transparent")
        p_frame.pack(fill="x", padx=16, pady=(4, 14))
        ctk.CTkLabel(p_frame, text="Password: ", font=("Segoe UI", 12, "bold"), text_color=COLOR_TEXT_DARK).pack(side="left")
        
        saved_pass = getattr(self, '_password', '••••••••')
        pass_lbl = ctk.CTkLabel(p_frame, text="••••••••", font=("Segoe UI", 12), text_color=COLOR_CARD_DARK)
        pass_lbl.pack(side="left")
        
        def toggle_password():
            if pass_lbl.cget("text") == "••••••••":
                pass_lbl.configure(text=saved_pass)
                toggle_btn.configure(text="🙈 Hide")
            else:
                pass_lbl.configure(text="••••••••")
                toggle_btn.configure(text="👁️ Show")

        toggle_btn = ctk.CTkButton(p_frame, text="👁️ Show", width=60, height=20,
                                   fg_color=COLOR_CARD_LIGHT, text_color=COLOR_TEXT_DARK,
                                   hover_color="#D9C2A3", command=toggle_password)
        toggle_btn.pack(side="left", padx=15)
        # ──────────────────────────────────────────────────────────────────────

        # Existing Layout Grid (Preferences & Notifications)
        row1 = ctk.CTkFrame(scroll, fg_color="transparent")
        row1.pack(fill="x", padx=32, pady=(0, 16))
        row1.columnconfigure(0, weight=1)
        row1.columnconfigure(1, weight=1)

        # Left: Preference Panel
        pref = ctk.CTkFrame(row1, fg_color=COLOR_CARD_BG, corner_radius=12,
                            border_width=1, border_color="#D9C2A3")
        pref.grid(row=0, column=0, sticky="nsew", padx=(0, 8))
        ctk.CTkLabel(pref, text="🎨 Preferences", font=("Segoe UI", 14, "bold"),
                     text_color=COLOR_TEXT_DARK, anchor="w").pack(anchor="w", padx=16, pady=(14, 10))
        
        for lbl, sub in [("Language", "English (UK)"), ("Theme", "Light Theme")]:
            r = ctk.CTkFrame(pref, fg_color="transparent")
            r.pack(fill="x", padx=16, pady=(0, 10))
            ctk.CTkLabel(r, text=lbl, font=("Segoe UI", 12, "bold"),
                         text_color=COLOR_TEXT_DARK, anchor="w").pack(anchor="w")
            ctk.CTkLabel(r, text=sub, font=("Segoe UI", 10),
                         text_color=COLOR_TEXT_LIGHT, anchor="w").pack(anchor="w")
        ctk.CTkFrame(pref, height=6, fg_color="transparent").pack()

        # Right: Notification Panel
        notif = ctk.CTkFrame(row1, fg_color=COLOR_CARD_BG, corner_radius=12,
                             border_width=1, border_color="#D9C2A3")
        notif.grid(row=0, column=1, sticky="nsew", padx=(8, 0))
        ctk.CTkLabel(notif, text="🔔 Notifications", font=("Segoe UI", 14, "bold"),
                     text_color=COLOR_TEXT_DARK, anchor="w").pack(anchor="w", padx=16, pady=(14, 10))
        
        for lbl, sub in [("Safety Alert", "Receive important safety alerts"),
                         ("Awareness update", "Receive updates and awareness tips")]:
            r = ctk.CTkFrame(notif, fg_color="transparent")
            r.pack(fill="x", padx=16, pady=(0, 10))
            ctk.CTkSwitch(r, text="", fg_color=COLOR_CARD_LIGHT,
                          progress_color=COLOR_ACCENT_ORANGE,
                          button_color=COLOR_SIDEBAR,
                          width=46, height=22).pack(side="left", padx=(0, 14))
            tc = ctk.CTkFrame(r, fg_color="transparent")
            tc.pack(side="left")
            ctk.CTkLabel(tc, text=lbl, font=("Segoe UI", 12, "bold"),
                         text_color=COLOR_TEXT_DARK, anchor="w").pack(anchor="w")
            ctk.CTkLabel(tc, text=sub, font=("Segoe UI", 10),
                         text_color=COLOR_TEXT_LIGHT, anchor="w").pack(anchor="w")
        ctk.CTkFrame(notif, height=6, fg_color="transparent").pack()

        # Save Changes Button
        save_btn = ctk.CTkButton(scroll, text="💾 Save Changes",
                                 font=("Georgia", 14, "bold"),
                                 fg_color=COLOR_ACCENT_ORANGE,
                                 hover_color="#A04810",
                                 text_color="white",
                                 corner_radius=10, height=46)
        save_btn.pack(fill="x", padx=32, pady=(0, 32))

        def save():
            save_btn.configure(text="✓ Saved!", fg_color="#4A8C30")
            scroll.after(2000, lambda: save_btn.configure(text="💾 Save Changes", fg_color=COLOR_ACCENT_ORANGE))

        save_btn.configure(command=save)
        return root_frame
# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    create_tables()
    app = SafePathApp()
    app.mainloop()